In [46]:
import sys
import re
from multistrand.objects import *
from multistrand.options import Options, Literals
from multistrand.system import SimSystem

In [47]:
def print_trajectory(o):
    seqstring=''
    for i in range(len(o.full_trajectory)): # go through each output microstate of the trajectory
        time = o.full_trajectory_times[i]   # time at which this microstate is entered
        states = o.full_trajectory[i]       # this is a list of the complexes present in this tube microstate
        newseqs = []
        for state in states: newseqs += [ state[3] ]   # extract the strand sequences in each complex (joined by "+" for multistranded complexes)
        newseqstring = ' '.join(newseqs)    # make a space-separated string of complexes, to represent the whole tube system sequence
        if not newseqstring == seqstring :
            print(newseqstring)
            seqstring=newseqstring          # because strand order can change upon association of dissociation, print it when it changes
        structs = []
        for state in states: structs += [ state[4] ]   # similarly extract the secondary structures for each complex
        tubestruct = ' '.join(structs)      # give the dot-paren secondary structure for the whole test tube
        dG=0
        for state in states: dG += state[5]
        print('%s t=%11.9f seconds, dG=%6.2f kcal/mol' % (tubestruct,time, dG))
        
        if i == len(o.full_trajectory) - 1001:
            print ("The followings are Checkpoint Reference")

In [48]:
def make_checkpoint_inputs(checkpoint, strandIncumbent, strandTarget, strandInvader):
    
    order2strand_dict = {'target': strandTarget, 
                     'incumbent': strandIncumbent, 
                     'invader': strandInvader
                     }
    
    if len(checkpoint) == 2:
        checkpoint_seed = checkpoint[0][0]
        checkpoint_structures_1 = checkpoint[0][4]
        checkpoint_structures_2 = checkpoint[1][4]
        
        checkpoint_order_1 =  re.findall(r'\d+:([a-zA-Z]+)', checkpoint[0][2])
        checkpoint_order_2 =  re.findall(r'\d+:([a-zA-Z]+)', checkpoint[1][2])
        
        if len(checkpoint_order_1) == 1:
            checkpoint_strand_1 = [order2strand_dict[checkpoint_order_1[0]]]
            checkpoint_strand_2 = [order2strand_dict[checkpoint_order_2[0]],
                                   order2strand_dict[checkpoint_order_2[1]],
                                   ]
            checkpoint_complex_1 = Complex(strands=checkpoint_strand_1, 
                                           structure=checkpoint_structures_1)
            checkpoint_complex_2 = Complex(strands=checkpoint_strand_2, 
                                           structure=checkpoint_structures_2)            
        else:
            checkpoint_strand_1 = [order2strand_dict[checkpoint_order_1[0]],
                                   order2strand_dict[checkpoint_order_1[1]],
                                   ]
            checkpoint_strand_2 = [order2strand_dict[checkpoint_order_2[0]]],
            checkpoint_complex_1 = Complex(strands=checkpoint_strand_1, 
                                           structure=checkpoint_structures_1)
            checkpoint_complex_2 = Complex(strands=checkpoint_strand_2)
                                   
        checkpoint_start_state= [checkpoint_complex_1, checkpoint_complex_2]
    
    else:
        checkpoint_seed = checkpoint[0][0]
        checkpoin_structures = checkpoint[0][4]
        checkpoin_order = [item.split(':')[1] for item in checkpoint[0][2].split(',')]
        checkpoin_strands = [order2strand_dict[checkpoin_order[0]], 
                             order2strand_dict[checkpoin_order[1]], 
                             order2strand_dict[checkpoin_order[2]]]
        checkpoint_start_state = [Complex(strands=checkpoin_strands, structure=checkpoin_structures)]
        
    return checkpoint_start_state, checkpoint_seed

In [49]:
def create_setup():
 
    toeholdSeq = "ATGTGGA"  # 7 nt toehold option

    incumbent = "TGGTGTTTGTGGGTGTGGTGAGTTTGAGGTTGA"
    target = "CCCTCCACATTCAACCTCAAACTCACC"
    invader = "GGTGAGTTTGAGGTTGAATGTGGA"
    invader = invader + toeholdSeq
    
    # set up the actual complexes
    strandIncumbent = Strand(name="incumbent", sequence=incumbent)
    strandTarget = Strand(name="target", sequence=target)
    strandInvader = Strand(name="invader", sequence=invader)
    
    intialDotParen = '.' * 16 + '(' * 17 + "+" + '.' * 10 + ')' * 17  
    intialInvaderDotParen = '.' * len(invader)
    successDotParen = '.' * 33
    
    initialComplex = Complex(strands=[strandIncumbent, strandTarget], structure=intialDotParen)
    initialInvader = Complex(strands=[strandInvader], structure=intialInvaderDotParen)
    successComplex = Complex(strands=[strandIncumbent], structure=successDotParen)
    
    stopSuccess = StopCondition(Literals.success, [(successComplex, Literals.exact_macrostate, 0)])
    
    # set up config
    o = Options(
        simulation_mode="Trajectory",
        substrate_type="DNA",
        num_simulations=1, 
        # simulation_time=float('inf'),
        simulation_time=1e-2,
        dangles="Some", 
        temperature=4, 
        join_concentration = 100e-6, # 100 uM
        gt_enable = False,
        output_interval = 1, # record every 1000 steps
        verbosity=0,
        start_state = [initialComplex, initialInvader],
        stop_conditions = [stopSuccess]
        )
    o.DNA23Metropolis()
    
    return o, strandIncumbent, strandTarget, strandInvader, stopSuccess    
    
        

In [50]:
def create_setup_2(checkpoint_start_state,stopSuccess,checkpoint_seed):
     
    # set up config
    o = Options(
        simulation_mode="Trajectory",
        substrate_type="DNA",
        num_simulations=1, 
        # simulation_time=float('inf'),
        simulation_time=1e-2,
        dangles="Some", 
        temperature=4, 
        join_concentration = 100e-6, # 100 uM
        gt_enable = False,
        output_interval = 1,
        verbosity=0,
        start_state = checkpoint_start_state,
        stop_conditions = [stopSuccess],
        initial_seed = checkpoint_seed
        )
    o.DNA23Metropolis()
    
    return o

### Generate the entire trajectory as reference

In [51]:
o1, strandIncumbent, strandTarget, strandInvader, stopSuccess = create_setup()
s1 = SimSystem(o1)
s1.start()

In [52]:
stdoutOrigin=sys.stdout 
sys.stdout = open(f"./o1.txt", "w")
print_trajectory(o1)
sys.stdout.flush()  # Flush the output here
sys.stdout.close()
sys.stdout=stdoutOrigin

### Generate checkpoint for restart the trajectory

In [62]:
checkpoint = o1.full_trajectory[50]
checkpoint

[(-6416507279180419254,
  1,
  '5:invader',
  'GGTGAGTTTGAGGTTGAATGTGGAATGTGGA',
  '..........(....(....)....).....',
  4.477686603256488,
  -7.121239722714816),
 (-6416507279180419254,
  0,
  '3:incumbent,4:target',
  'TGGTGTTTGTGGGTGTGGTGAGTTTGAGGTTGA+CCCTCCACATTCAACCTCAAACTCACC',
  '................(((((((((((((((((+..........)))))))))))))))))',
  -27.43832932437477,
  -135.42123972271483)]

In [63]:
checkpoint_start_state, checkpoint_seed = make_checkpoint_inputs(checkpoint, strandIncumbent, strandTarget, strandInvader)

In [64]:
o2 = create_setup_2(checkpoint_start_state,stopSuccess,checkpoint_seed)
s2 = SimSystem(o2)
s2.start()

In [65]:
stdoutOrigin=sys.stdout 
sys.stdout = open(f"./o2.txt", "w")
print_trajectory(o2)
sys.stdout.flush()  # Flush the output here
sys.stdout.close()
sys.stdout=stdoutOrigin